# Задание 2: Ранжирование BM25 и оценка результатов поиска

### Коллекция WikIR

Используем подмножество **en1k** коллекции [WikIR](https://github.com/getalp/wikIR) (Frej et al., 2020).

WikIR — автоматически построенный IR-датасет на основе английской Wikipedia:

- **Документы** — тексты статей Wikipedia (заголовки удалены), отфильтрованные по минимальной длине 200 токенов.
- **Запросы** — заголовки статей Wikipedia.
- **Оценки релевантности** получены из структуры Wikipedia:
  - **relevance = 2** — документ извлечён из той же статьи, что и запрос (точное соответствие);
  - **relevance = 1** — статья документа содержит гиперссылку на статью запроса (тематическая связь).

**Источник:** Frej, J., Schwab, D., & Chevallet, J.-P. (2020). *WIKIR: A Python Toolkit for Building a Large-scale Wikipedia-based English Information Retrieval Dataset.* LREC 2020. https://aclanthology.org/2020.lrec-1.237/

### Форматы TREC

Для оценки IR-систем используются два стандартных формата (из конференции TREC — Text REtrieval Conference):

**qrels** — файл с «правильными ответами» (какие документы релевантны каким запросам). 4 колонки через табуляцию:

**runs** — файл с результатами работы поисковой системы. 6 колонок:

In [17]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import ir_measures
from ir_measures import AP, P, nDCG

In [2]:
DATA_DIR = "../HW_1/data/wikIR1k"

# Загрузка тестовых запросов
queries = pd.read_csv(f"{DATA_DIR}/test/queries.csv")
queries.columns = ["query_id", "text"]
queries.head(10)

,query_id,text
0,158491,southern methodist university
1,5728,halakha
2,13554,chief justice of the united states
3,32674,patsy cline
4,406391,dierks bentley
5,5115,goidelic languages
6,15469,johns hopkins university
7,62953,kalmar county
8,152444,manawatu wanganui
9,104086,bulacan


In [3]:
# Загрузка qrels
qrels = pd.read_csv(
    f"{DATA_DIR}/test/qrels",
    sep="\t",
    header=None,
    names=["query_id", "iter", "doc_id", "relevance"],
)

qrels.head()

,query_id,iter,doc_id,relevance
0,158491,0,158491,2
1,158491,0,2130828,1
2,158491,0,730939,1
3,158491,0,1666627,1
4,158491,0,2102124,1


In [4]:
qrels["relevance"].unique()

array([2, 1])

## Задание 1: Базовая статистика по тестовым запросам

In [5]:
# 1. Количество запросов
n_queries = len(queries)
print(f"Количество тестовых запросов: {n_queries}")

Количество тестовых запросов: 100


In [6]:
# 2. Длина запросов в словах
queries["word_count"] = queries["text"].str.split().str.len()

print("Длина запросов в словах:")
print(queries["word_count"].describe().to_string())

print("\nРаспределение длин:")
print(queries["word_count"].value_counts().sort_index().to_string())

Длина запросов в словах:
count    100.000000
mean       2.070000
std        1.216511
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        9.000000

Распределение длин:
word_count
1    32
2    46
3    14
4     5
6     2
9     1


In [7]:
# 3. Количество релевантных документов на запрос
# нерелевантные документы отсутствуют
rel_per_query = qrels.groupby("query_id").size()

print("Количество релевантных документов на запрос:")
print(rel_per_query.describe().to_string())
print("\nРаспределение:")
print(rel_per_query.value_counts().sort_index().head(20).to_string())

Количество релевантных документов на запрос:
count     100.000000
mean       44.350000
std       275.103888
min         6.000000
25%         7.000000
50%         9.000000
75%        19.250000
max      2759.000000

Распределение:
6     18
7     16
8     13
9      4
10     7
11     6
12     2
14     2
15     2
16     2
18     2
19     1
20     3
21     2
22     1
23     1
24     2
26     2
28     1
30     1


In [8]:
# Сводная таблица
summary = pd.DataFrame({
    "Метрика": [
        "Количество запросов",
        "Средняя длина запроса (слов)",
        "Медианная длина запроса (слов)",
        "Мин. длина запроса (слов)",
        "Макс. длина запроса (слов)",
        "Среднее кол-во релевантных док. на запрос",
        "Медианное кол-во релевантных док. на запрос",
        "Мин. кол-во релевантных док. на запрос",
        "Макс. кол-во релевантных док. на запрос",
    ],
    "Значение": [
        n_queries,
        round(queries["word_count"].mean(), 2),
        queries["word_count"].median(),
        queries["word_count"].min(),
        queries["word_count"].max(),
        round(rel_per_query.mean(), 2),
        rel_per_query.median(),
        rel_per_query.min(),
        rel_per_query.max(),
    ],
})
summary

,Метрика,Значение
0,Количество запросов,100.00
1,Средняя длина запроса (слов),2.07
2,Медианная длина запроса (слов),2.00
3,Мин. длина запроса (слов),1.00
4,Макс. длина запроса (слов),9.00
5,Среднее кол-во релевантных док. на запрос,44.35
6,Медианное кол-во релевантных док. на запрос,9.00
7,Мин. кол-во релевантных док. на запрос,6.00
8,Макс. кол-во релевантных док. на запрос,2759.00


## Задание 2: Поиск с TF-IDF и BM25 по трём вариантам коллекции

Запускаем тестовые запросы на трёх вариантах коллекции:
- **original** — текст без обработки
- **stemmed** — стемминг (NLTK PorterStemmer)
- **lemmatized** — лемматизация (spaCy `en_core_web_sm`)

Для каждого варианта используем два метода ранжирования:
- **TF-IDF** (sklearn TfidfVectorizer + cosine similarity)
- **BM25** (rank_bm25)

Итого 6 прогонов. Результаты сохраняем в формате TREC runs.

In [9]:
# Загрузка документов
documents = pd.read_csv(f"{DATA_DIR}/documents.csv")
documents.columns = ["doc_id", "text"]
print(f"Документов: {len(documents)}")

Документов: 369721


In [10]:
# Функции предобработки текста

stemmer = PorterStemmer()

def stem_text(text: str) -> str:
    return " ".join(stemmer.stem(w) for w in text.split())

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize_text(text: str) -> str:
    doc = nlp(text)
    return " ".join(token.lemma_ for token in doc)

In [11]:
# Подготовка трёх вариантов коллекции документов (с независимым кешированием)
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

docs_original = documents["text"].tolist()
N_DOCS = len(docs_original)
LEMMA_N_PROCESS = max(1, (os.cpu_count() or 1) - 1)


def load_or_build_doc_variant(name: str, cache_path: Path, builder):
    if cache_path.exists():
        docs = pd.read_pickle(cache_path)
        if len(docs) == N_DOCS:
            print(f"{name}: загрузка из кеша {cache_path.name}")
            return docs

        print(
            f"{name}: кеш {cache_path.name} повреждён или устарел "
            f"({len(docs)} вместо {N_DOCS}), пересчёт..."
        )
    else:
        print(f"{name}: кеш {cache_path.name} не найден")

    docs = builder()
    pd.to_pickle(docs, cache_path)
    return docs


docs_stemmed = load_or_build_doc_variant(
    "Stemmed",
    CACHE_DIR / "docs_stemmed.pkl",
    lambda: [stem_text(t) for t in tqdm(docs_original, desc="Stemming")],
)

docs_lemmatized = load_or_build_doc_variant(
    "Lemmatized",
    CACHE_DIR / "docs_lemmatized.pkl",
    lambda: [
        " ".join(token.lemma_ for token in doc)
        for doc in tqdm(
            nlp.pipe(
                docs_original,
                batch_size=2000,
                n_process=LEMMA_N_PROCESS,
            ),
            total=N_DOCS,
            desc=f"Lemmatizing ({LEMMA_N_PROCESS} proc)",
        )
    ],
)

print(f"original={N_DOCS}, stemmed={len(docs_stemmed)}, lemmatized={len(docs_lemmatized)}")

Stemmed: кеш docs_stemmed.pkl не найден


Stemming: 100%|██████████| 369721/369721 [05:24<00:00, 1139.88it/s]


Lemmatized: кеш docs_lemmatized.pkl не найден


Lemmatizing (7 proc): 100%|██████████| 369721/369721 [43:24<00:00, 141.93it/s]  


Готово: original=369721, stemmed=369721, lemmatized=369721


In [13]:
# Подготовка трёх вариантов запросов

queries_original = queries["text"].tolist()
queries_stemmed = [stem_text(t) for t in queries_original]
queries_lemmatized = [
    " ".join(token.lemma_ for token in doc)
    for doc in nlp.pipe(queries_original, batch_size=256)
]

query_ids = queries["query_id"].tolist()
doc_ids = documents["doc_id"].tolist()

collection_variants = {
    "original": {"docs": docs_original, "queries": queries_original},
    "stemmed": {"docs": docs_stemmed, "queries": queries_stemmed},
    "lemmatized": {"docs": docs_lemmatized, "queries": queries_lemmatized},
}

pd.DataFrame(
    {
        "variant": list(collection_variants.keys()),
        "n_docs": [len(v["docs"]) for v in collection_variants.values()],
        "n_queries": [len(v["queries"]) for v in collection_variants.values()],
    }
)

,variant,n_docs,n_queries
0,original,369721,100
1,stemmed,369721,100
2,lemmatized,369721,100


In [14]:
TOP_K = 100  # сохраняем top-100 документов на запрос

RUNS_DIR = Path("runs")
RUNS_DIR.mkdir(exist_ok=True)


def save_trec_run(query_ids, doc_ids, all_scores, run_id, top_k=TOP_K):
    """Сохраняет результаты ранжирования в формате TREC run."""
    lines = []
    for row_idx, qid in enumerate(query_ids):
        scores = all_scores[row_idx]
        top_indices = np.argsort(scores)[::-1][:top_k]
        for rank, doc_idx in enumerate(top_indices, start=1):
            lines.append(
                f"{qid} Q0 {doc_ids[doc_idx]} {rank} {scores[doc_idx]:.6f} {run_id}"
            )

    path = RUNS_DIR / f"{run_id}.txt"
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")

    print(f"Сохранено: {path} ({len(lines)} строк)")
    return path


def run_tfidf_experiment(variant_name, docs, queries):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(docs)

    start = time.perf_counter()
    query_vectors = vectorizer.transform(queries)
    scores = cosine_similarity(query_vectors, tfidf_matrix)
    query_time = time.perf_counter() - start

    run_id = f"tfidf_{variant_name}"
    run_path = save_trec_run(query_ids, doc_ids, scores, run_id)
    return {
        "model": "tfidf",
        "variant": variant_name,
        "query_time_sec": query_time,
        "run_path": str(run_path),
    }


def run_bm25_experiment(variant_name, docs, queries):
    bm25 = BM25Okapi([doc.split() for doc in docs])

    start = time.perf_counter()
    scores = np.vstack([bm25.get_scores(query.split()) for query in queries])
    query_time = time.perf_counter() - start

    run_id = f"bm25_{variant_name}"
    run_path = save_trec_run(query_ids, doc_ids, scores, run_id)
    return {
        "model": "bm25",
        "variant": variant_name,
        "query_time_sec": query_time,
        "run_path": str(run_path),
    }

In [15]:
# TF-IDF на трёх вариантах коллекции

experiment_results = []

for variant_name, payload in collection_variants.items():
    print(f"\n--- TF-IDF | {variant_name} ---")
    result = run_tfidf_experiment(
        variant_name,
        payload["docs"],
        payload["queries"],
    )
    experiment_results.append(result)
    print(f"Query time: {result['query_time_sec']:.3f} s")


--- TF-IDF | original ---
Сохранено: runs/tfidf_original.txt (10000 строк)
Query time: 0.492 s

--- TF-IDF | stemmed ---
Сохранено: runs/tfidf_stemmed.txt (10000 строк)
Query time: 0.432 s

--- TF-IDF | lemmatized ---
Сохранено: runs/tfidf_lemmatized.txt (10000 строк)
Query time: 0.441 s


In [16]:
# Задание 2. BM25 на трёх вариантах коллекции

for variant_name, payload in collection_variants.items():
    print(f"\n--- BM25 | {variant_name} ---")
    result = run_bm25_experiment(
        variant_name,
        payload["docs"],
        payload["queries"],
    )
    experiment_results.append(result)
    print(f"Query time: {result['query_time_sec']:.3f} s")

results_df = pd.DataFrame(experiment_results)
results_df = results_df.sort_values(["model", "variant"]).reset_index(drop=True)
results_df


--- BM25 | original ---
Сохранено: runs/bm25_original.txt (10000 строк)
Query time: 16.610 s

--- BM25 | stemmed ---
Сохранено: runs/bm25_stemmed.txt (10000 строк)
Query time: 16.878 s

--- BM25 | lemmatized ---
Сохранено: runs/bm25_lemmatized.txt (10000 строк)
Query time: 16.214 s


,model,variant,query_time_sec,run_path
0,bm25,lemmatized,16.214494,runs/bm25_lemmatized.txt
1,bm25,original,16.609807,runs/bm25_original.txt
2,bm25,stemmed,16.878019,runs/bm25_stemmed.txt
3,tfidf,lemmatized,0.441311,runs/tfidf_lemmatized.txt
4,tfidf,original,0.491833,runs/tfidf_original.txt
5,tfidf,stemmed,0.431887,runs/tfidf_stemmed.txt


`TF-IDF` показал значительно более высокую скорость выполнения запросов, чем `BM25`: около `0.43-0.49 с` против `16.21-16.88 с` на весь набор тестовых запросов. Среди вариантов предобработки лучший результат по времени у `TF-IDF` показал `stemmed`, а у `BM25` `lemmatized`, однако различия между вариантами предобработки невелики по сравнению с разницей между самими моделями.

## Задание 3: Оценка runs с помощью ir-measures

In [19]:
# Метрики и их параметры
# P(rel=1, judged_only=False)@k: документ считается релевантным при relevance >= 1
# AP(rel=1, judged_only=False): MAP считается как среднее AP по всем запросам
# nDCG(dcg='log2', judged_only=False)@20: graded relevance по qrels со срезом top-20

measures = [
    P(rel=1, judged_only=False) @ 1,
    P(rel=1, judged_only=False) @ 10,
    P(rel=1, judged_only=False) @ 20,
    AP(rel=1, judged_only=False),
    nDCG(dcg="log2", judged_only=False) @ 20,
]

In [20]:
qrels_ir = list(ir_measures.read_trec_qrels(f"{DATA_DIR}/test/qrels"))
evaluator = ir_measures.evaluator(measures, qrels_ir)

evaluation_rows = []

for row in results_df.itertuples(index=False):
    run = list(ir_measures.read_trec_run(row.run_path))
    scores = evaluator.calc_aggregate(run)
    evaluation_rows.append(
        {
            "model": row.model,
            "variant": row.variant,
            "query_time_sec": row.query_time_sec,
            "P@1": scores[P(rel=1, judged_only=False) @ 1],
            "P@10": scores[P(rel=1, judged_only=False) @ 10],
            "P@20": scores[P(rel=1, judged_only=False) @ 20],
            "MAP": scores[AP(rel=1, judged_only=False)],
            "nDCG@20": scores[nDCG(dcg="log2", judged_only=False) @ 20],
        }
    )

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df = evaluation_df.sort_values(["MAP", "nDCG@20"], ascending=False).reset_index(drop=True)
evaluation_df

,model,variant,query_time_sec,P@1,P@10,P@20,MAP,nDCG@20
0,bm25,lemmatized,16.214494,0.48,0.210,0.1480,0.170629,0.356117
1,bm25,original,16.609807,0.49,0.212,0.1500,0.170297,0.356960
2,bm25,stemmed,16.878019,0.49,0.209,0.1455,0.169325,0.354021
3,tfidf,original,0.491833,0.51,0.204,0.1345,0.160425,0.349401
4,tfidf,lemmatized,0.441311,0.48,0.201,0.1295,0.156274,0.338231
5,tfidf,stemmed,0.431887,0.50,0.194,0.1285,0.154880,0.335431


По качеству ранжирования лучшие результаты в целом показал `BM25`: он дал наибольшие значения `P@10`, `P@20`, `MAP` и `nDCG@20`. Лучшим вариантом по интегральным метрикам оказался `bm25_original` (`P@10 = 0.212`, `P@20 = 0.150`, `MAP = 0.170297`, `nDCG@20 = 0.356960`), а `bm25_lemmatized` показал практически такой же уровень качества и даже немного лучший `MAP = 0.170629`. Вариант `bm25_stemmed` оказался немного слабее двух остальных версий BM25.

`TF-IDF` уступает `BM25` почти по всем метрикам глубины ранжирования, однако показал лучший результат по `P@1`: максимальное значение у `tfidf_original` (`0.51`), что означает более частое точное попадание первого документа в релевантный. При этом на более глубоких срезах (`P@10`, `P@20`, `MAP`, `nDCG@20`) `TF-IDF` стабильно хуже `BM25`.

## Задание 4: Анализ результатов

In [ ]:
# Сравнение систем по разным метрикам

metric_rank_summary = evaluation_df[["model", "variant", "P@1", "P@10", "P@20", "MAP", "nDCG@20"]].copy()
for metric in ["P@1", "P@10", "P@20", "MAP", "nDCG@20"]:
    metric_rank_summary[f"rank_{metric}"] = metric_rank_summary[metric].rank(ascending=False, method="min")

metric_rank_summary.sort_values(["rank_MAP", "rank_nDCG@20", "rank_P@1"]).reset_index(drop=True)

,model,variant,P@1,P@10,P@20,MAP,nDCG@20,rank_P@1,rank_P@10,rank_P@20,rank_MAP,rank_nDCG@20
0,bm25,lemmatized,0.48,0.210,0.1480,0.170629,0.356117,5.0,2.0,2.0,1.0,2.0
1,bm25,original,0.49,0.212,0.1500,0.170297,0.356960,3.0,1.0,1.0,2.0,1.0
2,bm25,stemmed,0.49,0.209,0.1455,0.169325,0.354021,3.0,3.0,3.0,3.0,3.0
3,tfidf,original,0.51,0.204,0.1345,0.160425,0.349401,1.0,4.0,4.0,4.0,4.0
4,tfidf,lemmatized,0.48,0.201,0.1295,0.156274,0.338231,5.0,5.0,5.0,5.0,5.0
5,tfidf,stemmed,0.50,0.194,0.1285,0.154880,0.335431,2.0,6.0,6.0,6.0,6.0


In [22]:
# Влияние морфологической обработки относительно original

morphology_effect = evaluation_df.copy()
original_scores = (
    morphology_effect[morphology_effect["variant"] == "original"]
    .set_index("model")[["P@1", "P@10", "P@20", "MAP", "nDCG@20"]]
)

for metric in ["P@1", "P@10", "P@20", "MAP", "nDCG@20"]:
    morphology_effect[f"delta_{metric}_vs_original"] = morphology_effect.apply(
        lambda row: row[metric] - original_scores.loc[row["model"], metric],
        axis=1,
    )

morphology_effect.sort_values(["model", "variant"]).reset_index(drop=True)

,model,variant,query_time_sec,P@1,P@10,P@20,MAP,nDCG@20,delta_P@1_vs_original,delta_P@10_vs_original,delta_P@20_vs_original,delta_MAP_vs_original,delta_nDCG@20_vs_original
0,bm25,lemmatized,16.214494,0.48,0.210,0.1480,0.170629,0.356117,-0.01,-0.002,-0.0020,0.000332,-0.000843
1,bm25,original,16.609807,0.49,0.212,0.1500,0.170297,0.356960,0.00,0.000,0.0000,0.000000,0.000000
2,bm25,stemmed,16.878019,0.49,0.209,0.1455,0.169325,0.354021,0.00,-0.003,-0.0045,-0.000972,-0.002939
3,tfidf,lemmatized,0.441311,0.48,0.201,0.1295,0.156274,0.338231,-0.03,-0.003,-0.0050,-0.004151,-0.011169
4,tfidf,original,0.491833,0.51,0.204,0.1345,0.160425,0.349401,0.00,0.000,0.0000,0.000000,0.000000
5,tfidf,stemmed,0.431887,0.50,0.194,0.1285,0.154880,0.335431,-0.01,-0.010,-0.0060,-0.005545,-0.013969


In [24]:
# Easy / hard queries и различия между BM25 и TF-IDF на original

queries_analysis = queries[["query_id", "text", "word_count"]].copy()
queries_analysis["query_id"] = queries_analysis["query_id"].astype(str)

topic_measures = [
    P(rel=1, judged_only=False) @ 1,
    AP(rel=1, judged_only=False),
    nDCG(dcg="log2", judged_only=False) @ 20,
]
topic_measure_names = {
    P(rel=1, judged_only=False) @ 1: "P@1",
    AP(rel=1, judged_only=False): "AP",
    nDCG(dcg="log2", judged_only=False) @ 20: "nDCG@20",
}


def calc_topic_metrics(run_path: str, run_name: str) -> pd.DataFrame:
    run = list(ir_measures.read_trec_run(run_path))
    rows = [
        {
            "query_id": metric.query_id,
            "measure": topic_measure_names.get(metric.measure, str(metric.measure)),
            "value": metric.value,
        }
        for metric in ir_measures.iter_calc(topic_measures, qrels_ir, run)
    ]
    df = pd.DataFrame(rows).pivot(index="query_id", columns="measure", values="value").reset_index()
    df.columns.name = None
    df["run"] = run_name
    return df


topic_bm25_original = calc_topic_metrics("runs/bm25_original.txt", "bm25_original")
topic_tfidf_original = calc_topic_metrics("runs/tfidf_original.txt", "tfidf_original")

easy_queries = (
    topic_bm25_original
    .merge(queries_analysis, on="query_id", how="left")
    .sort_values(["AP", "nDCG@20"], ascending=False)
    .head(10)
)

hard_queries = (
    topic_bm25_original
    .merge(queries_analysis, on="query_id", how="left")
    .sort_values(["AP", "nDCG@20"], ascending=True)
    .head(10)
)

topic_compare = (
    topic_bm25_original.merge(
        topic_tfidf_original,
        on="query_id",
        suffixes=("_bm25", "_tfidf"),
    )
    .merge(queries_analysis, on="query_id", how="left")
)

topic_compare["AP_diff_bm25_minus_tfidf"] = (
    topic_compare["AP_bm25"]
    - topic_compare["AP_tfidf"]
)

bm25_advantage_queries = topic_compare.sort_values("AP_diff_bm25_minus_tfidf", ascending=False).head(10)
tfidf_advantage_queries = topic_compare.sort_values("AP_diff_bm25_minus_tfidf", ascending=True).head(10)

easy_queries

,query_id,AP,P@1,nDCG@20,run,text,word_count
63,32674,0.722951,1.0,0.624178,bm25_original,patsy cline,2
88,75295,0.679167,1.0,0.849280,bm25_original,dave matthews band,3
56,267973,0.598958,1.0,0.822220,bm25_original,gambela region,2
11,121384,0.564583,1.0,0.654766,bm25_original,supertramp,1
6,113000,0.532275,1.0,0.748263,bm25_original,sulfide,1
96,87686,0.506349,1.0,0.646022,bm25_original,evesham,1
2,104086,0.445832,1.0,0.716428,bm25_original,bulacan,1
71,422206,0.416593,1.0,0.580538,bm25_original,enugu state,2
25,14410,0.395970,1.0,0.730094,bm25_original,xml,1
92,8140,0.373512,1.0,0.658935,bm25_original,michael moorcock,2


In [25]:
hard_queries

,query_id,AP,P@1,nDCG@20,run,text,word_count
37,1897191,0.000000,0.0,0.000000,bm25_original,centrism,1
94,83078,0.000000,0.0,0.000000,bm25_original,pomacentridae,1
20,13540,0.002128,0.0,0.230308,bm25_original,united kingdom,2
87,738573,0.002244,0.0,0.000000,bm25_original,television presenter,2
89,79032,0.006944,0.0,0.000000,bm25_original,elijah wood,2
40,206019,0.007353,0.0,0.096826,bm25_original,billboard 200,2
90,7911,0.010101,0.0,0.106173,bm25_original,medicine,1
79,641464,0.011483,0.0,0.099292,bm25_original,brisbane central business district,4
76,5622,0.011759,0.0,0.040379,bm25_original,homer,1
31,158491,0.021514,0.0,0.114580,bm25_original,southern methodist university,3


In [26]:
bm25_advantage_queries[[
    "query_id", "text", "word_count",
    "AP_bm25",
    "AP_tfidf",
    "AP_diff_bm25_minus_tfidf",
]]

,query_id,text,word_count,AP_bm25,AP_tfidf,AP_diff_bm25_minus_tfidf
88,75295,dave matthews band,3,0.679167,0.313049,0.366118
78,62953,kalmar county,2,0.337745,0.138889,0.198856
62,32559,yes minister,2,0.215000,0.026786,0.188214
69,406391,dierks bentley,2,0.263935,0.085899,0.178036
38,196387,bordeaux wine official classification of 1855,6,0.187865,0.033333,0.154532
56,267973,gambela region,2,0.598958,0.449236,0.149722
39,200237,yvonne de carlo,3,0.169526,0.026161,0.143365
99,996687,rom cartridge,2,0.297778,0.174074,0.123704
34,172877,michael bolton,2,0.281593,0.168843,0.112750
66,362197,texas motor speedway,3,0.250000,0.138889,0.111111


In [27]:
tfidf_advantage_queries[[
    "query_id", "text", "word_count",
    "AP_bm25",
    "AP_tfidf",
    "AP_diff_bm25_minus_tfidf",
]]

,query_id,text,word_count,AP_bm25,AP_tfidf,AP_diff_bm25_minus_tfidf
49,25130,paul gauguin,2,0.179599,0.536111,-0.356512
54,25983,franz liszt,2,0.115801,0.343017,-0.227216
33,16952,nelly furtado,2,0.262925,0.482444,-0.219519
63,32674,patsy cline,2,0.722951,0.937925,-0.214974
46,22343,mick jagger,2,0.206843,0.367652,-0.160809
86,73752,jackson browne,2,0.102407,0.237013,-0.134606
24,139082,roadrunner records,2,0.036218,0.147959,-0.111741
26,145194,volos,1,0.083333,0.181818,-0.098485
4,107590,west bromwich,2,0.068027,0.155280,-0.087252
92,8140,michael moorcock,2,0.373512,0.458333,-0.084821


In [28]:
# 4.4. Влияние длины запроса

query_length_effect = (
    topic_bm25_original
    .merge(queries_analysis[["query_id", "word_count"]], on="query_id", how="left")
    .groupby("word_count")[[
        "P@1",
        "AP",
        "nDCG@20",
    ]]
    .mean()
    .round(4)
)

query_length_effect

,P@1,AP,nDCG@20
word_count,,,
1,0.4688,0.1709,0.3588
2,0.5000,0.1687,0.3484
3,0.5000,0.1807,0.3767
4,0.4000,0.1080,0.3169
6,0.5000,0.2255,0.4392
9,1.0000,0.2820,0.4512


In [29]:
topic_bm25_with_len = topic_bm25_original.merge(
    queries_analysis[["query_id", "word_count"]], on="query_id", how="left"
)

topic_bm25_with_len[[
    "word_count",
    "P@1",
    "AP",
    "nDCG@20",
]].corr(numeric_only=True).round(4)

,word_count,P@1,AP,nDCG@20
word_count,1.0000,0.0590,0.0396,0.0503
P@1,0.0590,1.0000,0.5077,0.6984
AP,0.0396,0.5077,1.0000,0.8639
nDCG@20,0.0503,0.6984,0.8639,1.0000


In [30]:
# 4.5. Примеры top-ranked документов, отсутствующих в qrels

qrels_pairs = set(zip(qrels["query_id"].astype(str), qrels["doc_id"].astype(str)))
doc_text_map = dict(zip(documents["doc_id"].astype(str), documents["text"]))
queries_text_map = dict(zip(queries_analysis["query_id"], queries_analysis["text"]))

run_bm25_original_df = pd.read_csv(
    "runs/bm25_original.txt",
    sep=r"\s+",
    header=None,
    names=["query_id", "Q0", "doc_id", "rank", "score", "run_id"],
)
run_bm25_original_df["query_id"] = run_bm25_original_df["query_id"].astype(str)
run_bm25_original_df["doc_id"] = run_bm25_original_df["doc_id"].astype(str)

nonrelevant_examples = []
for query_id, group in run_bm25_original_df.groupby("query_id"):
    query_text = queries_text_map[query_id]
    for row in group.head(5).itertuples(index=False):
        if (row.query_id, row.doc_id) not in qrels_pairs:
            doc_text = doc_text_map.get(row.doc_id, "")
            nonrelevant_examples.append(
                {
                    "query_id": row.query_id,
                    "query": query_text,
                    "rank": row.rank,
                    "doc_id": row.doc_id,
                    "score": round(row.score, 3),
                    "snippet": " ".join(doc_text.split()[:40]),
                }
            )
            break

pd.DataFrame(nonrelevant_examples).head(10)

,query_id,query,rank,doc_id,score,snippet
0,101626,edmonton eskimos,2,1684402,25.342,a total of 18 players were selected as territo...
1,10166,palestinian national authority,1,472366,22.879,these political objectives include self determ...
2,104086,bulacan,5,1800147,14.164,gregorio del pilar responsible for the eventua...
3,104453,uniting church in australia,1,615389,35.402,the oldest congregational church was founded i...
4,107590,west bromwich,1,1441161,19.880,west bromwich returned to the championship aft...
5,108909,arad county,2,1076780,18.078,his parents were isaia and floarea goldi the f...
6,113000,sulfide,3,525590,17.919,it occurs in nature as the dark indigo blue mi...
7,113827,4th arrondissement of paris,2,2226640,32.027,he emigrated to france as a child during world...
8,11534,saarland,1,1981252,17.977,it falls into the category of geographic tlds ...
9,116217,atari games,2,450625,21.166,from 2004 to 2011 since 2011 the consoles have...


Есть и лёгкие, и трудные запросы: лучше всего ранжируются конкретные именованные сущности, хуже всего короткие и неоднозначные запросы. Одной метрики недостаточно: `TF-IDF` лучший по `P@1`, но `BM25` лучше по `P@10`, `P@20`, `MAP` и `nDCG@20`.

Морфологическая обработка почти не помогла: вариант `original` обычно лучший или почти лучший, а `stemmed` чаще ухудшает качество. Длина запроса влияет слабо, а многие документы вне `qrels` попадают в топ из-за сильного лексического совпадения с запросом.

## Задание 5: Подбор параметров `k1` и `b` для BM25 на validation set

In [33]:
# Validation set

validation_queries = pd.read_csv(f"{DATA_DIR}/validation/queries.csv")
validation_queries.columns = ["query_id", "text"]
validation_queries["query_id"] = validation_queries["query_id"].astype(str)

validation_qrels_ir = list(ir_measures.read_trec_qrels(f"{DATA_DIR}/validation/qrels"))

# Оптимизируем MAP как основную метрику качества ранжирования
bm25_opt_measure = AP(rel=1, judged_only=False)

# Сетка параметров для формального подбора
k1_grid = [0.6, 0.9, 1.2, 1.5, 1.8, 2.1]
b_grid = [0.1, 0.3, 0.5, 0.75, 0.9]

BM25_TUNING_CACHE = CACHE_DIR / "bm25_validation_grid.pkl"

In [34]:
def build_bm25_run_df(query_df, bm25, doc_ids, run_id, top_k=TOP_K):
    rows = []
    for query_id, query_text in query_df[["query_id", "text"]].itertuples(index=False):
        scores = bm25.get_scores(str(query_text).split())
        top_indices = np.argsort(scores)[::-1][:top_k]
        for rank, doc_idx in enumerate(top_indices, start=1):
            rows.append(
                {
                    "query_id": str(query_id),
                    "doc_id": str(doc_ids[doc_idx]),
                    "score": float(scores[doc_idx]),
                    "rank": rank,
                    "run_id": run_id,
                }
            )
    return pd.DataFrame(rows)


def tune_bm25_on_validation(tokenized_docs, doc_ids, validation_queries, validation_qrels_ir):
    grid_rows = []
    for k1 in tqdm(k1_grid, desc="k1 grid"):
        for b in b_grid:
            bm25 = BM25Okapi(tokenized_docs, k1=k1, b=b)
            val_run_df = build_bm25_run_df(
                validation_queries,
                bm25,
                doc_ids,
                run_id=f"bm25_val_k1_{k1}_b_{b}",
            )
            score = ir_measures.calc_aggregate(
                [bm25_opt_measure],
                validation_qrels_ir,
                val_run_df,
            )[bm25_opt_measure]
            grid_rows.append({"k1": k1, "b": b, "MAP": score})
    return pd.DataFrame(grid_rows)


tokenized_docs_original = [doc.split() for doc in docs_original]

if BM25_TUNING_CACHE.exists():
    bm25_grid_results = pd.read_pickle(BM25_TUNING_CACHE)
else:
    bm25_grid_results = tune_bm25_on_validation(
        tokenized_docs_original,
        doc_ids,
        validation_queries,
        validation_qrels_ir,
    )
    pd.to_pickle(bm25_grid_results, BM25_TUNING_CACHE)

bm25_grid_results = bm25_grid_results.sort_values(["MAP", "k1", "b"], ascending=[False, True, True]).reset_index(drop=True)
bm25_grid_results.head(10)

k1 grid: 100%|██████████| 6/6 [16:35<00:00, 165.90s/it]


,k1,b,MAP
0,0.6,0.50,0.152493
1,0.6,0.10,0.152492
2,0.6,0.90,0.152422
3,0.6,0.75,0.152416
4,0.6,0.30,0.152354
5,0.9,0.50,0.151593
6,0.9,0.75,0.151474
7,0.9,0.10,0.151474
8,0.9,0.30,0.151414
9,0.9,0.90,0.151373


In [35]:
bm25_best_params = bm25_grid_results.iloc[0].to_dict()
bm25_default_params = {"k1": 1.5, "b": 0.75}

bm25_validation_summary = pd.DataFrame(
    [
        {
            "setting": "default",
            "k1": bm25_default_params["k1"],
            "b": bm25_default_params["b"],
            "MAP": bm25_grid_results.loc[
                (bm25_grid_results["k1"] == bm25_default_params["k1"])
                & (bm25_grid_results["b"] == bm25_default_params["b"]),
                "MAP",
            ].iloc[0],
        },
        {
            "setting": "best_on_validation",
            "k1": bm25_best_params["k1"],
            "b": bm25_best_params["b"],
            "MAP": bm25_best_params["MAP"],
        },
    ]
)

bm25_validation_summary

,setting,k1,b,MAP
0,default,1.5,0.75,0.149766
1,best_on_validation,0.6,0.50,0.152493


In [36]:
# Сравнение default BM25 и tuned BM25 на test set

test_bm25_measures = [
    P(rel=1, judged_only=False) @ 1,
    P(rel=1, judged_only=False) @ 10,
    P(rel=1, judged_only=False) @ 20,
    AP(rel=1, judged_only=False),
    nDCG(dcg="log2", judged_only=False) @ 20,
]


def evaluate_bm25_setting_on_test(setting_name, k1, b):
    bm25 = BM25Okapi(tokenized_docs_original, k1=float(k1), b=float(b))
    run_df = build_bm25_run_df(
        queries.assign(query_id=queries["query_id"].astype(str)),
        bm25,
        doc_ids,
        run_id=f"bm25_{setting_name}_k1_{k1}_b_{b}",
    )

    run_path = RUNS_DIR / f"bm25_{setting_name}_k1_{k1}_b_{b}.txt"
    run_df_for_save = run_df[["query_id", "doc_id", "rank", "score", "run_id"]].copy()
    run_df_for_save.insert(1, "Q0", "Q0")
    with open(run_path, "w") as f:
        for row in run_df_for_save.itertuples(index=False):
            f.write(f"{row.query_id} {row.Q0} {row.doc_id} {row.rank} {row.score:.6f} {row.run_id}\n")

    agg = ir_measures.calc_aggregate(
        test_bm25_measures,
        qrels_ir,
        run_df,
    )
    return {
        "setting": setting_name,
        "k1": float(k1),
        "b": float(b),
        "run_path": str(run_path),
        "P@1": agg[P(rel=1, judged_only=False) @ 1],
        "P@10": agg[P(rel=1, judged_only=False) @ 10],
        "P@20": agg[P(rel=1, judged_only=False) @ 20],
        "MAP": agg[AP(rel=1, judged_only=False)],
        "nDCG@20": agg[nDCG(dcg="log2", judged_only=False) @ 20],
    }


bm25_test_comparison = pd.DataFrame(
    [
        evaluate_bm25_setting_on_test("default", 1.5, 0.75),
        evaluate_bm25_setting_on_test("tuned", bm25_best_params["k1"], bm25_best_params["b"]),
    ]
)

bm25_test_comparison

,setting,k1,b,run_path,P@1,P@10,P@20,MAP,nDCG@20
0,default,1.5,0.75,runs/bm25_default_k1_1.5_b_0.75.txt,0.49,0.212,0.1500,0.170297,0.356960
1,tuned,0.6,0.50,runs/bm25_tuned_k1_0.6_b_0.5.txt,0.50,0.206,0.1475,0.167883,0.352668


Лучшая пара параметров на `validation set` по `MAP` это `k1 = 0.6`, `b = 0.5`: она дала `MAP = 0.152493` против `0.149766` у параметров по умолчанию. Однако на `test set` tuned-версия оказалась хуже дефолтной по `MAP`, `P@10`, `P@20` и `nDCG@20`, улучшив только `P@1`.

Значит, выигрыш на validation не перенёсся на test, а стандартные параметры `BM25` для этого датасета оказались более устойчивыми.